In [1]:
import os, glob

DATASET_ROOT = "/kaggle/input/v7-pytorch-for-hybrid"
print("Exists:", os.path.exists(DATASET_ROOT))
print("Folders:", os.listdir(DATASET_ROOT))

print("\nSample train images:", len(glob.glob(DATASET_ROOT + "/train/images/*")))
print("Sample train labels:", len(glob.glob(DATASET_ROOT + "/train/labels/*")))
print("data.yaml exists:", os.path.exists(DATASET_ROOT + "/data.yaml"))


Exists: True
Folders: ['README.dataset.txt', 'README.roboflow.txt', 'data.yaml', 'valid', 'test', 'train']

Sample train images: 6499
Sample train labels: 6499
data.yaml exists: True


In [2]:
import shutil, os

SRC = "/kaggle/input/v7-pytorch-for-hybrid"
DST = "/kaggle/working/roadsign_data"

if not os.path.exists(DST):
    shutil.copytree(SRC, DST)

print("Copied to:", DST)
print(os.listdir(DST))

Copied to: /kaggle/working/roadsign_data
['README.dataset.txt', 'README.roboflow.txt', 'train', 'data.yaml', 'test', 'valid']


In [3]:
# This scans YOUR notebook working directory for any "PY" heredoc artifacts (not perfect, but helps)
import os, glob

print("Searching for suspicious files in /kaggle/working ...")
hits = []
for fp in glob.glob("/kaggle/working/**/*", recursive=True):
    if os.path.isfile(fp) and fp.endswith((".py", ".txt", ".sh")):
        try:
            t = open(fp, "r", encoding="utf-8", errors="ignore").read()
            if "python - <<" in t or "wanted `PY`" in t or "\nPY\n" in t:
                hits.append(fp)
        except:
            pass

print("Found:", len(hits))
for h in hits[:20]:
    print(h)


Searching for suspicious files in /kaggle/working ...
Found: 0


In [4]:
import os, shutil, torch

print("CUDA available:", torch.cuda.is_available())
print("Torch:", torch.__version__)

os.chdir("/kaggle/working")

# fresh clone
if os.path.exists("yolov7"):
    shutil.rmtree("yolov7")

!git clone --depth 1 https://github.com/WongKinYiu/yolov7.git
%cd /kaggle/working/yolov7

# Minimal safe deps (won't touch numpy/torch)
!pip -q install --no-cache-dir pyyaml tqdm protobuf tensorboard psutil thop requests matplotlib pandas

# Download YOLOv7 pretrained
!wget -q https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt -O yolov7.pt

print("yolov7.pt exists:", os.path.exists("yolov7.pt"))


CUDA available: True
Torch: 2.8.0+cu126
Cloning into 'yolov7'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 127 (delta 25), reused 123 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 39.53 MiB | 22.93 MiB/s, done.
Resolving deltas: 100% (25/25), done.
/kaggle/working/yolov7
yolov7.pt exists: True


In [5]:
import os, yaml

BASE = "/kaggle/working/roadsign_data"   # you already copied dataset here

in_yaml = os.path.join(BASE, "data.yaml")
assert os.path.exists(in_yaml), f"Not found: {in_yaml}"

with open(in_yaml, "r") as f:
    d = yaml.safe_load(f)

# overwrite with absolute paths
d["train"] = os.path.join(BASE, "train/images")
d["val"]   = os.path.join(BASE, "valid/images")

# optional test
test_path = os.path.join(BASE, "test/images")
if os.path.exists(test_path):
    d["test"] = test_path

out_yaml = "/kaggle/working/roadsign_data_kaggle.yaml"
with open(out_yaml, "w") as f:
    yaml.safe_dump(d, f, sort_keys=False)

print("Saved:", out_yaml)
print("train:", d["train"])
print("val  :", d["val"])
print("num classes:", d.get("nc", "missing"))
print("names count:", len(d.get("names", [])))


Saved: /kaggle/working/roadsign_data_kaggle.yaml
train: /kaggle/working/roadsign_data/train/images
val  : /kaggle/working/roadsign_data/valid/images
num classes: 29
names count: 29


In [6]:
import os
os.environ["WANDB_MODE"] = "disabled"

In [7]:
import pathlib, re

train_path = pathlib.Path("/kaggle/working/yolov7/train.py")
txt = train_path.read_text()

# Patch the specific line that crashes:
# torch.load(weights, map_location=device)
txt_new = txt.replace(
    "torch.load(weights, map_location=device)",
    "torch.load(weights, map_location=device, weights_only=False)"
)

if txt_new == txt:
    print("Patch NOT applied (line not found). We'll search and patch with regex.")
    # regex fallback
    txt_new = re.sub(
        r"torch\.load\((weights\s*,\s*map_location=device)\)",
        r"torch.load(\1, weights_only=False)",
        txt
    )

train_path.write_text(txt_new)
print("Patched train.py successfully ✅")


Patched train.py successfully ✅


In [8]:
%cd /kaggle/working/yolov7

!python train.py \
  --img 640 \
  --batch 16 \
  --epochs 100 \
  --data /kaggle/working/roadsign_data_kaggle.yaml \
  --weights /kaggle/working/yolov7/yolov7.pt \
  --name roadsign_yolov7 \
  --project /kaggle/working/runs/train \
  --device 0 \
  --workers 2 \
  --exist-ok


/kaggle/working/yolov7
2026-01-21 00:28:56.758180: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768955336.962955      62 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768955337.020176      62 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768955337.512156      62 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768955337.512207      62 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768955337.512214      62 computation_placer.cc:177]

In [9]:
import pathlib, re

gen_path = pathlib.Path("/kaggle/working/yolov7/utils/general.py")
txt = gen_path.read_text()

# Find and patch the line inside strip_optimizer:
# x = torch.load(f, map_location=torch.device('cpu'))
txt_new = txt.replace(
    "x = torch.load(f, map_location=torch.device('cpu'))",
    "x = torch.load(f, map_location=torch.device('cpu'), weights_only=False)"
)

if txt_new == txt:
    print("Direct replace not found, trying regex patch...")
    txt_new = re.sub(
        r"x\s*=\s*torch\.load\(\s*f\s*,\s*map_location=torch\.device\('cpu'\)\s*\)",
        "x = torch.load(f, map_location=torch.device('cpu'), weights_only=False)",
        txt
    )

gen_path.write_text(txt_new)
print("Patched utils/general.py successfully ✅")


Patched utils/general.py successfully ✅


In [10]:
import os, glob

run_dir = "/kaggle/working/runs/train/roadsign_yolov7"
print("run_dir exists:", os.path.exists(run_dir))

print("\nContents of run_dir:")
if os.path.exists(run_dir):
    for p in sorted(os.listdir(run_dir))[:50]:
        print(" -", p)

print("\nWeights files:")
for p in glob.glob(run_dir + "/weights/*.pt"):
    print(p)


run_dir exists: True

Contents of run_dir:
 - F1_curve.png
 - PR_curve.png
 - P_curve.png
 - R_curve.png
 - confusion_matrix.png
 - events.out.tfevents.1768955360.2df7272008f9.62.0
 - hyp.yaml
 - opt.yaml
 - results.png
 - results.txt
 - test_batch0_labels.jpg
 - test_batch0_pred.jpg
 - test_batch1_labels.jpg
 - test_batch1_pred.jpg
 - test_batch2_labels.jpg
 - test_batch2_pred.jpg
 - train_batch0.jpg
 - train_batch1.jpg
 - train_batch2.jpg
 - train_batch3.jpg
 - train_batch4.jpg
 - train_batch5.jpg
 - train_batch6.jpg
 - train_batch7.jpg
 - train_batch8.jpg
 - train_batch9.jpg
 - weights

Weights files:
/kaggle/working/runs/train/roadsign_yolov7/weights/last.pt
/kaggle/working/runs/train/roadsign_yolov7/weights/best.pt
/kaggle/working/runs/train/roadsign_yolov7/weights/epoch_095.pt
/kaggle/working/runs/train/roadsign_yolov7/weights/epoch_049.pt
/kaggle/working/runs/train/roadsign_yolov7/weights/epoch_074.pt
/kaggle/working/runs/train/roadsign_yolov7/weights/epoch_000.pt
/kaggle/workin

In [11]:
%cd /kaggle/working/yolov7
!python train.py --resume /kaggle/working/runs/train/roadsign_yolov7/weights/last.pt --device 0


/kaggle/working/yolov7
2026-01-21 11:00:22.000190: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768993222.022154     124 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768993222.028880     124 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768993222.047114     124 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768993222.047145     124 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768993222.047149     124 computation_placer.cc:177]

In [12]:
import pathlib, re, glob, os

# 1) Patch datasets.py (labels.cache load)
ds_path = pathlib.Path("/kaggle/working/yolov7/utils/datasets.py")
txt = ds_path.read_text()
txt2 = re.sub(
    r"torch\.load\(\s*cache_path\s*\)",
    "torch.load(cache_path, weights_only=False)",
    txt
)
ds_path.write_text(txt2)
print("Patched datasets.py ✅")

# 2) Patch experimental.py (attempt_load / inference load)
exp_path = pathlib.Path("/kaggle/working/yolov7/models/experimental.py")
txt = exp_path.read_text()
txt2 = txt.replace(
    "ckpt = torch.load(w, map_location=map_location)",
    "ckpt = torch.load(w, map_location=map_location, weights_only=False)"
)
exp_path.write_text(txt2)
print("Patched experimental.py ✅")

# 3) Delete old cache files (so it rebuilds safely)
cache_files = glob.glob("/kaggle/working/roadsign_data/**/labels.cache", recursive=True)
for f in cache_files:
    os.remove(f)
print("Deleted cache files:", len(cache_files))


Patched datasets.py ✅
Patched experimental.py ✅
Deleted cache files: 2


In [13]:
import os
print("best.pt:", os.path.exists("/kaggle/working/runs/train/roadsign_yolov7/weights/best.pt"))
print("last.pt:", os.path.exists("/kaggle/working/runs/train/roadsign_yolov7/weights/last.pt"))


best.pt: True
last.pt: True


In [14]:
%cd /kaggle/working/yolov7

BEST = "/kaggle/working/runs/train/roadsign_yolov7/weights/best.pt"
SOURCE = "/kaggle/working/roadsign_data/valid/images"

!python detect.py \
  --weights {BEST} \
  --source {SOURCE} \
  --img-size 640 \
  --conf 0.25 \
  --save-txt \
  --save-conf \
  --project /kaggle/working/runs/detect \
  --name yolov7_valid_preds \
  --exist-ok

print("\n✅ Saved to: /kaggle/working/runs/detect/yolov7_valid_preds")


/kaggle/working/yolov7
Namespace(weights=['/kaggle/working/runs/train/roadsign_yolov7/weights/best.pt'], source='/kaggle/working/roadsign_data/valid/images', img_size=640, conf_thres=0.25, iou_thres=0.45, device='', view_img=False, save_txt=True, save_conf=True, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project='/kaggle/working/runs/detect', name='yolov7_valid_preds', exist_ok=True, no_trace=False)
YOLOR 🚀 a207844 torch 2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269.25MB)

Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
/usr/local/lib/python3.12/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
Model Summary: 306 layers, 36630706 parameters, 6194944 gradients, 103.6

In [15]:
import os, glob
from PIL import Image
import yaml

# Paths
BASE = "/kaggle/working/roadsign_data"
IMG_DIR = f"{BASE}/train/images"
LBL_DIR = f"{BASE}/train/labels"
OUT_CROP_ROOT = "/kaggle/working/crops_train"

os.makedirs(OUT_CROP_ROOT, exist_ok=True)

# Load class names from your fixed YAML
with open("/kaggle/working/roadsign_data_kaggle.yaml", "r") as f:
    y = yaml.safe_load(f)
names = y["names"]

def yolo_to_xyxy(w, h, xc, yc, bw, bh):
    x1 = (xc - bw/2) * w
    y1 = (yc - bh/2) * h
    x2 = (xc + bw/2) * w
    y2 = (yc + bh/2) * h
    x1 = int(max(0, x1)); y1 = int(max(0, y1))
    x2 = int(min(w-1, x2)); y2 = int(min(h-1, y2))
    return x1, y1, x2, y2

img_paths = glob.glob(os.path.join(IMG_DIR, "*"))
saved = 0
skipped = 0

for ip in img_paths:
    stem = os.path.splitext(os.path.basename(ip))[0]
    lp = os.path.join(LBL_DIR, stem + ".txt")
    if not os.path.exists(lp):
        continue

    try:
        img = Image.open(ip).convert("RGB")
    except:
        skipped += 1
        continue

    w, h = img.size
    with open(lp, "r") as f:
        lines = [x.strip() for x in f.readlines() if x.strip()]

    for i, line in enumerate(lines):
        parts = line.split()
        if len(parts) < 5:
            continue

        cls = int(float(parts[0]))
        xc, yc, bw, bh = map(float, parts[1:5])

        x1, y1, x2, y2 = yolo_to_xyxy(w, h, xc, yc, bw, bh)
        if (x2 - x1) < 12 or (y2 - y1) < 12:
            continue

        crop = img.crop((x1, y1, x2, y2))

        cls_name = names[cls]
        out_dir = os.path.join(OUT_CROP_ROOT, cls_name)
        os.makedirs(out_dir, exist_ok=True)

        out_path = os.path.join(out_dir, f"{stem}_{i}.jpg")
        crop.save(out_path, quality=95)
        saved += 1

print("✅ Crops saved:", saved)
print("⚠️ Images skipped (read error):", skipped)
print("✅ Class folders created:", len([d for d in os.listdir(OUT_CROP_ROOT) if os.path.isdir(os.path.join(OUT_CROP_ROOT,d))]))
print("📁 crops path:", OUT_CROP_ROOT)


✅ Crops saved: 6555
⚠️ Images skipped (read error): 0
✅ Class folders created: 29
📁 crops path: /kaggle/working/crops_train


In [16]:
import os, torch, torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

CROPS = "/kaggle/working/crops_train"
OUT_DIR = "/kaggle/working/hybrid_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

# Data transforms
train_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.2),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# Dataset
full = datasets.ImageFolder(CROPS, transform=train_tfms)
num_classes = len(full.classes)

# Split train/val
val_ratio = 0.15
val_size = int(len(full) * val_ratio)
train_size = len(full) - val_size
train_ds, val_ds = random_split(full, [train_size, val_size])

# IMPORTANT: val needs val transforms
val_ds.dataset.transform = val_tfms

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| classes:", num_classes, "| train:", len(train_ds), "| val:", len(val_ds))

# Model
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

best_acc = 0.0
best_path = os.path.join(OUT_DIR, "resnet50_best.pth")

EPOCHS = 30  # keep small first; you can increase later (10-20)
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    # Validate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out = model(x)
            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    train_loss = running_loss / len(train_ds)
    val_acc = correct / total

    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.4f} | val_acc={val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "classes": full.classes
        }, best_path)
        print("✅ Saved best ->", best_path)

print("✅ Best val_acc:", best_acc)
print("✅ Saved file:", best_path, "exists:", os.path.exists(best_path))


Device: cuda | classes: 29 | train: 5572 | val: 983
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 198MB/s]


Epoch 1/30 | train_loss=0.4329 | val_acc=1.0000
✅ Saved best -> /kaggle/working/hybrid_artifacts/resnet50_best.pth
Epoch 2/30 | train_loss=0.0087 | val_acc=0.9980
Epoch 3/30 | train_loss=0.0014 | val_acc=1.0000
Epoch 4/30 | train_loss=0.0097 | val_acc=1.0000
Epoch 5/30 | train_loss=0.0215 | val_acc=0.9949
Epoch 6/30 | train_loss=0.0113 | val_acc=0.9990
Epoch 7/30 | train_loss=0.0164 | val_acc=0.9786
Epoch 8/30 | train_loss=0.0121 | val_acc=0.9949
Epoch 9/30 | train_loss=0.0058 | val_acc=1.0000
Epoch 10/30 | train_loss=0.0068 | val_acc=1.0000
Epoch 11/30 | train_loss=0.0093 | val_acc=0.9990
Epoch 12/30 | train_loss=0.0012 | val_acc=1.0000
Epoch 13/30 | train_loss=0.0048 | val_acc=0.9990
Epoch 14/30 | train_loss=0.0103 | val_acc=1.0000
Epoch 15/30 | train_loss=0.0092 | val_acc=1.0000
Epoch 16/30 | train_loss=0.0198 | val_acc=1.0000
Epoch 17/30 | train_loss=0.0014 | val_acc=1.0000
Epoch 18/30 | train_loss=0.0010 | val_acc=1.0000
Epoch 19/30 | train_loss=0.0002 | val_acc=1.0000
Epoch 20/30

In [17]:
import os, glob

print("Exists hybrid_artifacts:", os.path.exists("/kaggle/working/hybrid_artifacts"))
print("Files:", glob.glob("/kaggle/working/hybrid_artifacts/*")[:50])
print("hybrid_infer.py exists:", os.path.exists("/kaggle/working/hybrid_artifacts/hybrid_infer.py"))


Exists hybrid_artifacts: True
Files: ['/kaggle/working/hybrid_artifacts/resnet50_best.pth']
hybrid_infer.py exists: False


In [18]:
import sys
sys.path.insert(0, "/kaggle/working/hybrid_artifacts")

from hybrid_infer import HybridDetector


ModuleNotFoundError: No module named 'hybrid_infer'

In [ ]:
import pickle
with open("/kaggle/working/hybrid_artifacts/hybrid_bundle.pkl", "rb") as f:
    obj = pickle.load(f)


In [ ]:
%%writefile /kaggle/working/hybrid_artifacts/hybrid_infer.py
# hybrid_final.py
import os
import sys
from typing import List, Dict, Any, Tuple

import cv2
import numpy as np
from PIL import Image

import torch
import torch.nn.functional as F
from torchvision import models, transforms

# ----------------------------
# Config: ImageNet transforms
# ----------------------------
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# ----------------------------
# YOLOv7 helpers (requires YOLOv7 repo)
# ----------------------------
def _add_yolov7_to_syspath(yolov7_root: str):
    yolov7_root = os.path.abspath(yolov7_root)
    if yolov7_root not in sys.path:
        sys.path.insert(0, yolov7_root)

def _load_yolov7_model(yolov7_root: str, weights_path: str, device: str):
    _add_yolov7_to_syspath(yolov7_root)

    from models.experimental import attempt_load  # type: ignore

    model = attempt_load(weights_path, map_location=device)
    model.eval()
    return model

def _preprocess_for_yolov7(bgr: np.ndarray, img_size: int, stride: int):
    """
    Returns:
      img: torch tensor (1,3,h,w) normalized to 0..1
      ratio, (dw, dh) used for scaling coords back
      img0: original image
      img_letterboxed: letterboxed image
    """
    from utils.datasets import letterbox  # type: ignore

    img0 = bgr
    img_letterboxed, ratio, (dw, dh) = letterbox(img0, new_shape=img_size, stride=stride, auto=True)

    # BGR -> RGB, HWC -> CHW
    img = img_letterboxed[:, :, ::-1].transpose(2, 0, 1)
    img = np.ascontiguousarray(img)
    img = torch.from_numpy(img).float() / 255.0
    if img.ndimension() == 3:
        img = img.unsqueeze(0)
    return img, ratio, (dw, dh), img0, img_letterboxed

def _yolo_infer_scaled_dets(
    model,
    bgr: np.ndarray,
    img_size: int = 640,
    conf_thres: float = 0.25,
    iou_thres: float = 0.45,
    device: str = "cpu",
    use_half: bool = True
) -> List[List[float]]:
    """
    Runs YOLOv7 and returns dets_scaled:
      [[x1,y1,x2,y2, conf, cls_id], ...] scaled to original image coords.
    """
    from utils.general import non_max_suppression, scale_coords  # type: ignore

    stride = int(model.stride.max()) if hasattr(model, "stride") else 32

    img, _, _, img0, _ = _preprocess_for_yolov7(bgr, img_size, stride)
    img = img.to(device)

    half_ok = (device != "cpu") and use_half
    if half_ok:
        img = img.half()
        model.half()
    else:
        model.float()

    with torch.no_grad():
        pred = model(img, augment=False)[0]
        pred = non_max_suppression(pred, conf_thres, iou_thres)

    dets_scaled: List[List[float]] = []
    if pred[0] is None or len(pred[0]) == 0:
        return dets_scaled

    det = pred[0]
    # scale coords from letterboxed img shape back to original img shape
    det[:, :4] = scale_coords(img.shape[2:], det[:, :4], img0.shape).round()

    # det format: x1,y1,x2,y2, conf, cls
    for *xyxy, conf, cls_id in det.tolist():
        x1, y1, x2, y2 = xyxy
        dets_scaled.append([x1, y1, x2, y2, float(conf), float(cls_id)])
    return dets_scaled

# ----------------------------
# Hybrid Detector (Option B)
# ----------------------------
class HybridDetector:
    def __init__(
        self,
        yolov7_root: str,
        yolo_weights_path: str,
        resnet_ckpt_path: str,
        class_names: List[str],
        device: str = None,
        img_size: int = 640,
        conf_thres: float = 0.25,
        iou_thres: float = 0.45,
        confirm_thr: float = 0.60,
        override_thr: float = 0.85,
        pad_ratio: float = 0.10,        # expand box by 10% before crop
        use_half: bool = True
    ):
        self.class_names = class_names
        self.num_classes = len(class_names)

        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.img_size = img_size
        self.conf_thres = conf_thres
        self.iou_thres = iou_thres
        self.confirm_thr = confirm_thr
        self.override_thr = override_thr
        self.pad_ratio = pad_ratio
        self.use_half = use_half

        # YOLOv7
        self.yolo = _load_yolov7_model(yolov7_root, yolo_weights_path, self.device)

        # ResNet50
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.resnet.fc = torch.nn.Linear(self.resnet.fc.in_features, self.num_classes)

        ckpt = torch.load(resnet_ckpt_path, map_location="cpu")
        # allow both formats: pure state_dict OR dict with key 'model'
        if isinstance(ckpt, dict) and "model" in ckpt and isinstance(ckpt["model"], dict):
            state = ckpt["model"]
        else:
            state = ckpt
        self.resnet.load_state_dict(state, strict=True)

        self.resnet.to(self.device).eval()

        self.resnet_tfms = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def _expand_box(self, x1, y1, x2, y2, w, h) -> Tuple[int, int, int, int]:
        bw = x2 - x1
        bh = y2 - y1
        padx = int(bw * self.pad_ratio)
        pady = int(bh * self.pad_ratio)
        nx1 = max(0, x1 - padx)
        ny1 = max(0, y1 - pady)
        nx2 = min(w - 1, x2 + padx)
        ny2 = min(h - 1, y2 + pady)
        return nx1, ny1, nx2, ny2

    def _batch_resnet_predict(self, crops_bgr: List[np.ndarray]) -> Tuple[List[int], List[float]]:
        """
        Batched ResNet inference.
        Returns (pred_indices, pred_probs)
        """
        if len(crops_bgr) == 0:
            return [], []

        batch_tensors = []
        for crop_bgr in crops_bgr:
            crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
            pil = Image.fromarray(crop_rgb)
            batch_tensors.append(self.resnet_tfms(pil))

        x = torch.stack(batch_tensors, dim=0).to(self.device)
        with torch.no_grad():
            logits = self.resnet(x)
            probs = F.softmax(logits, dim=1)
            prob_vals, idxs = torch.max(probs, dim=1)

        return idxs.detach().cpu().tolist(), prob_vals.detach().cpu().tolist()

    def _apply_option_b(
        self,
        yolo_cls: int,
        resnet_cls: int,
        resnet_prob: float
    ) -> Tuple[int, str]:
        """
        Option B decision logic:
        - If same class and prob >= confirm_thr => confirmed (keep)
        - If different and prob >= override_thr => overridden (use ResNet)
        - Else keep YOLO
        """
        if resnet_cls == yolo_cls and resnet_prob >= self.confirm_thr:
            return yolo_cls, "confirmed"
        if resnet_cls != yolo_cls and resnet_prob >= self.override_thr:
            return resnet_cls, "overridden"
        return yolo_cls, "kept_yolo"

    def predict(self, image_bgr: np.ndarray) -> List[Dict[str, Any]]:
        """
        Input: original BGR image (cv2.imread)
        Output: list of results with final labels.
        """
        h, w = image_bgr.shape[:2]

        # 1) YOLO detections
        dets_scaled = _yolo_infer_scaled_dets(
            model=self.yolo,
            bgr=image_bgr,
            img_size=self.img_size,
            conf_thres=self.conf_thres,
            iou_thres=self.iou_thres,
            device=self.device,
            use_half=self.use_half
        )

        if len(dets_scaled) == 0:
            return []

        # 2) Prepare crops (with padding)
        crops = []
        meta = []  # store box + yolo info
        for x1, y1, x2, y2, conf, cls_id in dets_scaled:
            x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

            # clamp + sanity
            x1 = max(0, min(x1, w - 1))
            x2 = max(0, min(x2, w - 1))
            y1 = max(0, min(y1, h - 1))
            y2 = max(0, min(y2, h - 1))
            if x2 <= x1 or y2 <= y1:
                continue

            px1, py1, px2, py2 = self._expand_box(x1, y1, x2, y2, w, h)
            crop = image_bgr[py1:py2, px1:px2]
            if crop.size == 0:
                continue

            crops.append(crop)
            meta.append((x1, y1, x2, y2, float(conf), int(cls_id)))

        if len(crops) == 0:
            return []

        # 3) ResNet batch prediction
        resnet_idxs, resnet_probs = self._batch_resnet_predict(crops)

        # 4) Option B decision per detection
        results = []
        for (x1, y1, x2, y2, yolo_conf, yolo_cls), r_cls, r_prob in zip(meta, resnet_idxs, resnet_probs):
            final_cls, decision = self._apply_option_b(yolo_cls, r_cls, r_prob)

            results.append({
                "box": [x1, y1, x2, y2],
                "yolo_cls": yolo_cls,
                "yolo_name": self.class_names[yolo_cls],
                "yolo_conf": yolo_conf,
                "resnet_cls": r_cls,
                "resnet_name": self.class_names[r_cls],
                "resnet_prob": float(r_prob),
                "final_cls": final_cls,
                "final_name": self.class_names[final_cls],
                "decision": decision  # confirmed / overridden / kept_yolo
            })

        return results


# ----------------------------
# Simple CLI demo (optional)
# ----------------------------
if __name__ == "__main__":
    # Example usage:
    # python hybrid_final.py
    # (Edit paths below)
    yolov7_root = "./yolov7"
    yolo_weights = "./best.pt"
    resnet_weights = "./resnet50_best.pth"
    image_path = "./test.jpg"

    # class_names must match your data.yaml names order
    class_names = []  # TODO: paste your names list here

    det = HybridDetector(
        yolov7_root=yolov7_root,
        yolo_weights_path=yolo_weights,
        resnet_ckpt_path=resnet_weights,
        class_names=class_names,
        confirm_thr=0.60,
        override_thr=0.85
    )

    img = cv2.imread(image_path)
    out = det.predict(img)
    for r in out:
        print(r)


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/yolov7")
sys.path.insert(0, "/kaggle/working/hybrid_artifacts")

from hybrid_infer import HybridDetector


In [ ]:
import glob, random

YOLO_WEIGHTS = "/kaggle/working/runs/train/roadsign_yolov7/weights/best.pt"
RESNET_WEIGHTS = "/kaggle/working/hybrid_artifacts/resnet50_best.pth"
DATA_YAML = "/kaggle/working/roadsign_data_kaggle.yaml"

hd = HybridDetector(YOLO_WEIGHTS, RESNET_WEIGHTS, DATA_YAML)

valid_imgs = glob.glob("/kaggle/working/roadsign_data/valid/images/*")
demo_img = random.choice(valid_imgs)

res = hd.detect(demo_img, img_size=640, conf=0.25)
print("✅ Demo image:", demo_img)
print("✅ Detections:", len(res))
print("✅ First 3 results:", res[:3])


In [ ]:
import os, sys, glob, random

# Make sure YOLOv7 is importable
sys.path.append("/kaggle/working/yolov7")

# Make sure hybrid_infer.py folder is importable
sys.path.append("/kaggle/working/hybrid_artifacts")

os.chdir("/kaggle/working/yolov7")

from hybrid_infer import HybridDetector

YOLO_WEIGHTS = "/kaggle/working/runs/train/roadsign_yolov7/weights/best.pt"
RESNET_WEIGHTS = "/kaggle/working/hybrid_artifacts/resnet50_best.pth"
DATA_YAML = "/kaggle/working/roadsign_data_kaggle.yaml"

hd = HybridDetector(YOLO_WEIGHTS, RESNET_WEIGHTS, DATA_YAML)

valid_imgs = glob.glob("/kaggle/working/roadsign_data/valid/images/*")
demo_img = random.choice(valid_imgs)

res = hd.detect(demo_img, img_size=640, conf=0.25)
print("✅ Demo image:", demo_img)
print("✅ Detections:", len(res))
print("✅ First 3 results:", res[:3])


In [ ]:
import os, pickle, zipfile, yaml

ART_DIR = "/kaggle/working/hybrid_artifacts"
os.makedirs(ART_DIR, exist_ok=True)

YOLO_WEIGHTS = "/kaggle/working/runs/train/roadsign_yolov7/weights/best.pt"
RESNET_WEIGHTS = "/kaggle/working/hybrid_artifacts/resnet50_best.pth"
DATA_YAML = "/kaggle/working/roadsign_data_kaggle.yaml"
HYBRID_PY = "/kaggle/working/hybrid_artifacts/hybrid_infer.py"

with open(DATA_YAML, "r") as f:
    y = yaml.safe_load(f)

bundle = {
    "yolo_weights": YOLO_WEIGHTS,
    "resnet_weights": RESNET_WEIGHTS,
    "data_yaml": DATA_YAML,
    "class_names": y["names"],
    "hybrid_py": HYBRID_PY,
    "img_size": 640,
    "conf": 0.25,
    "iou": 0.45
}

pkl_path = os.path.join(ART_DIR, "hybrid_bundle.pkl")
with open(pkl_path, "wb") as f:
    pickle.dump(bundle, f)

print("✅ Saved:", pkl_path)

# Zip everything needed for Streamlit
zip_path = "/kaggle/working/hybrid_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(YOLO_WEIGHTS, arcname="best.pt")
    z.write(RESNET_WEIGHTS, arcname="resnet50_best.pth")
    z.write(HYBRID_PY, arcname="hybrid_infer.py")
    z.write(pkl_path, arcname="hybrid_bundle.pkl")

print("✅ ZIP ready:", zip_path)


In [ ]:
import zipfile, os, glob

pred_dir = "/kaggle/working/runs/detect/yolov7_valid_preds"
zip_path = "/kaggle/working/yolov7_valid_preds.zip"

files = glob.glob(pred_dir + "/**/*", recursive=True)
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        if os.path.isfile(f):
            z.write(f, arcname=os.path.relpath(f, pred_dir))

print("✅ Saved:", zip_path)
